# Ejercicios complementarios — Clase 5: Embeddings y Chunking

Esta notebook complementa `clase05_embeddings_chunking.ipynb`. No repite lo ya construido: propone variaciones, casos borde y mini-desafíos sobre los mismos módulos que quedaron persistidos en `ai_agent_project/src/ai_agent_course/`.

**Precondición.** Ya ejecutaste `clase05_embeddings_chunking.ipynb` completa y existen:

- `ai_agent_project/src/ai_agent_course/embeddings.py`
- `ai_agent_project/src/ai_agent_course/chunking.py`
- `ai_agent_project/src/ai_agent_course/retrieval.py`
- `ai_agent_project/artifacts/class05_report.json`

**Este material es opcional.** No forma parte del incremento obligatorio del proyecto (`ai_agent_project/`); es práctica adicional para profundizar. Ningún ejercicio modifica los módulos persistidos: trabajan sobre copias o funciones nuevas en esta misma notebook.


In [ ]:
# Verifica la precondición y deja el proyecto importable, igual que la notebook original.
from pathlib import Path
import json
import sys

PROJECT_DIR = Path("../ai_agent_project")
SRC_DIR = PROJECT_DIR / "src" / "ai_agent_course"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"

if str(PROJECT_DIR / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR / "src"))

required = [
    SRC_DIR / "embeddings.py",
    SRC_DIR / "chunking.py",
    SRC_DIR / "retrieval.py",
    ARTIFACTS_DIR / "class05_report.json",
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise RuntimeError(
        "Ejecutá clase05_embeddings_chunking.ipynb antes de esta práctica. Faltan: "
        + ", ".join(missing)
    )

class05_report = json.loads((ARTIFACTS_DIR / "class05_report.json").read_text(encoding="utf-8"))
print("Precondición verificada. Estrategia usada en clase:", class05_report["chunking"]["selected_strategy"])


## Ejercicio 1 — Impacto de `target_words` y `overlap_words` en el hit-rate

En la clase midieron `hit_rate` de `fixed`, `recursive` y `semantic` con `target_words=55`. Acá vas a repetir esa medición con chunks **más chicos** (`target_words=20`) y **más grandes** (`target_words=90`), manteniendo `recursive_chunks`, y comparar el `hit_rate` resultante contra el valor original de la clase.

**Pista.** Reconstruí el corpus con `DATA_DIR = PROJECT_DIR / "data"`, cargá `policies.json`, aplicá `document_chunks(policies, target_words=..., overlap_words=...)` e indexá con `InMemoryRetriever`. Reutilizá los mismos casos de prueba (consulta → `document_id` esperado) que en la clase.


In [ ]:
from ai_agent_course.chunking import document_chunks
from ai_agent_course.embeddings import build_embedding_provider
from ai_agent_course.retrieval import InMemoryRetriever

DATA_DIR = PROJECT_DIR / "data"
policies = json.loads((DATA_DIR / "policies.json").read_text(encoding="utf-8"))

eval_cases = [
    ("anticipación para vacaciones", "POL-VAC-001"),
    ("plazo de comprobantes de reintegro", "POL-EXP-002"),
    ("acciones por contraseña expuesta", "POL-SEC-003"),
    ("acuerdo para trabajo remoto", "POL-REM-004"),
]

embedding_provider = build_embedding_provider(False)

# TODO: completá la función para que indexe con el target_words/overlap_words recibidos
# y devuelva la proporción de casos donde el top-1 es el documento esperado.
def hit_rate_for(target_words: int, overlap_words: int) -> float:
    raise NotImplementedError("Implementá la evaluación de hit-rate para estos parámetros")

# TODO: llamá hit_rate_for con (20, 4) y con (90, 12) y compará contra el valor de la clase.
results = {
    # "target_20": hit_rate_for(20, 4),
    # "target_90": hit_rate_for(90, 12),
}
results


## Ejercicio 2 — Estadísticas de tamaño de chunk

Falta una función utilitaria que resuma el tamaño (en palabras) de una lista de chunks: mínimo, máximo y promedio. Es el tipo de chequeo rápido que usarías antes de indexar un corpus nuevo para detectar chunks anormalmente cortos o largos.

**Pista.** Cada chunk es un dict con clave `"text"`. Usá `len(chunk["text"].split())` para contar palabras.


In [ ]:
# TODO: implementá chunk_length_stats para que devuelva {"min": ..., "max": ..., "avg": ...}
def chunk_length_stats(chunks: list[dict]) -> dict:
    raise NotImplementedError

sample_chunks = document_chunks(policies, target_words=45, overlap_words=6)

stats = chunk_length_stats(sample_chunks)

# Auto-chequeo: no lo edites, sirve para validar tu implementación.
assert set(stats) == {"min", "max", "avg"}
assert stats["min"] <= stats["avg"] <= stats["max"]
print(stats)


## Ejercicio 3 — Caso borde: texto vacío o casi vacío

¿Qué pasa si un documento llega con `text=""` o con un texto de una sola palabra? Un pipeline de ingesta real recibe basura de vez en cuando y no debería romperse.

**Pista.** Probá `document_chunks` con un documento de texto vacío y otro de una sola palabra. Verificá cuántos chunks produce cada uno (¿alguno debería producir cero chunks?).


In [ ]:
edge_documents = [
    {
        "document_id": "EDGE-EMPTY",
        "source": "vacio.md",
        "page": 1,
        "text": "",
        "metadata": {"topic": "edge"},
    },
    {
        "document_id": "EDGE-ONEWORD",
        "source": "una_palabra.md",
        "page": 1,
        "text": "hola",
        "metadata": {"topic": "edge"},
    },
]

# TODO: aplicá document_chunks sobre edge_documents y guardá el resultado en edge_chunks.
edge_chunks = None

# TODO: descomentá y completá estos asserts una vez tengas edge_chunks.
# assert isinstance(edge_chunks, list)
# print("Chunks generados:", len(edge_chunks))
# for chunk in edge_chunks:
#     print(chunk["document_id"], repr(chunk["text"]))


## Ejercicio 4 — `fixed_size_chunks` vs `recursive_chunks` con abreviaciones

`recursive_chunks` separa oraciones con la expresión regular `(?<=[.!?])\s+`. Un texto con abreviaciones (`"Sr."`, `"Nº"`, `"art. 5"`) puede cortar oraciones en lugares raros porque el punto de la abreviación se interpreta como fin de oración.

**Pista.** Armá un texto corto con un par de abreviaciones y compará `fixed_size_chunks` contra `recursive_chunks` con el mismo `target_words`. ¿Cuál "corta peor" la oración?


In [ ]:
from ai_agent_course.chunking import fixed_size_chunks, recursive_chunks

abbreviation_text = (
    "El Sr. Gómez presentó la solicitud según el art. 12 del reglamento. "
    "La aprobación depende del área de RRHH y del presupuesto disponible este año."
)

# TODO: generá los chunks con ambas estrategias (target_words=12, overlap_words=2)
# e imprimilos para comparar visualmente dónde corta cada una.
fixed_result = None
recursive_result = None

# print("fixed:", fixed_result)
# print("recursive:", recursive_result)


## Ejercicio 5 — Falsos positivos léxicos en el embedding determinístico

`DeterministicSemanticEmbeddings` agrupa palabras relacionadas mediante un diccionario de sinónimos fijo (por ejemplo, `"licencia"` y `"descanso"` se mapean a `"vacaciones"`). Esto significa que dos textos sobre temas **distintos** pueden salir con similitud alta si comparten una palabra clave que el diccionario agrupa.

**Desafío.** Encontrá (o armá) un par de textos de temas distintos que produzcan una similitud coseno sospechosamente alta con este embedding, y explicá en un comentario por qué ocurre mirando el diccionario `synonyms` dentro de `embeddings.py`.


In [ ]:
import numpy as np

def cosine(a, b):
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    denom = float(np.linalg.norm(a) * np.linalg.norm(b))
    return 0.0 if denom == 0 else float(np.dot(a, b) / denom)

# TODO: reemplazá estos dos textos por un par que vos armes, de temas distintos,
# que comparta una palabra del diccionario de sinónimos de DeterministicSemanticEmbeddings.
text_a = "TODO: texto sobre un tema"
text_b = "TODO: texto sobre otro tema que comparte una palabra clave"

vectors = embedding_provider.embed_documents([text_a, text_b])
similarity = cosine(vectors[0], vectors[1])
print("Similitud:", round(similarity, 3))

# TODO: en un comentario, explicá por qué el diccionario de sinónimos produce este resultado.


## Cierre

Estos ejercicios no generan un reporte propio: son práctica sobre los módulos ya persistidos en `ai_agent_project/`. Si querés dejar evidencia de tu trabajo, guardá tus propias conclusiones en una celda markdown o en un archivo separado dentro de `ejercicios_complementarios/`, sin tocar `class05_report.json`.
